## Let's run intron clustering to annotate alternative splicing events given observed junctions in our cells 

In [1]:
!hostname

ne1dc6-005.nygenome.org


In [2]:
import os
import pandas as pd 
from sklearn.decomposition import TruncatedSVD
import anndata as ad
from scipy.sparse import coo_matrix

# turn this into AnnData object 
import anndata as ad
from scipy.sparse import csr_matrix
import numpy as np
import torch 
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc

from scipy.spatial.distance import cdist

import numpy as np

import sys
import os
import json
import numpy as np
import torch
import anndata as ad
from importlib import reload
from datetime import datetime
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import pyro 
import umap.umap_ as umap
import matplotlib.patches as mpatches
import scipy.sparse
import datetime
import sys
import random

sys.path.append('/gpfs/commons/home/kisaev/Leaflet-private/src/visualization')
# from visualize_ATSE import visualize_local_events

# Import custom modules
sys.path.append('/gpfs/commons/home/kisaev/Leaflet-private/src/beta-dirichlet-factor')

import factor_model
reload(factor_model)

import waypoints_prep_input as wayp
reload(wayp)

import full_leafletFA_pipeline_wALBF as LeafletFA

sys.path.append('/gpfs/commons/home/kisaev/Leaflet-private/src/clustering/')
import load_cluster_data as llc 

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

float_type = {"device": device, "dtype": torch.float}
if device == torch.device('cuda'):
    torch.set_default_tensor_type('torch.cuda.FloatTensor')

!hostname

2.3.0+cu121
12.1
2.3.0+cu121
12.1
2.3.0+cu121
12.1
2.3.0+cu121
12.1
Using device: cpu
Using device: cpu
Using device: cpu
ne1dc6-005.nygenome.org


#### Specify if we want to prep brain only file or ALL tissue file


In [3]:
brain_only=False
gtf_annot=False

#### Load merged anndata file containing all cells 

In [4]:
# input_file='/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/ATSE_Anndata_Object_20240927_214100.h5ad' 
input_file="/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/NOGTF/ATSE_Anndata_noGTF_Object_20241112_004133.h5ad"

adata_full = ad.read_h5ad(input_file)
splice_adata = adata_full.copy()
splice_adata.obs.reset_index(drop=True, inplace=True)
splice_adata.obs["cell_id_index"] = splice_adata.obs.index 

/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [5]:
# ATSE_file="/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/tabula_senis_test_intron_clusters_50_500000_10_20240927_single_cell.gz"
ATSE_file="/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/NOGTF/tabula_senis_annotationFREE_intron_clusters_50_500000_100_20241102_single_cell.gz"
atses = pd.read_csv(ATSE_file, sep="}")

In [6]:
atses.shape

(71500, 9)

In [7]:
if brain_only:
    splice_adata = splice_adata[splice_adata.obs["tissue"].isin(["Brain_Myeloid", "Brain_Non-Myeloid"])]
    splice_adata.obs.reset_index(drop=True, inplace=True)
    splice_adata.obs["cell_id_index"] = splice_adata.obs.index 
    print(splice_adata.obs.shape, splice_adata.obs.cell_id_index.max())

### Let's remove the 21m age group since so few cells not well represented... 

In [8]:
splice_adata = splice_adata[~(splice_adata.obs["age"] == "21m")]

In [9]:
splice_adata.obs["cell_clean"] = splice_adata.obs["cell_id"].str.replace(r'-(?=.*_)', '_', regex=True).values
splice_adata.obs["cell_clean"] = splice_adata.obs["cell_clean"].str.split('_').str[:2].str.join('_')

/scratch/ipykernel_1420882/2220754778.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  splice_adata.obs["cell_clean"] = splice_adata.obs["cell_id"].str.replace(r'-(?=.*_)', '_', regex=True).values
/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


### Load in gene expression matrix for smart-seq2 data and align cell IDs 

In [10]:
# Load the expression data (.h5ad file)
exp_file = "/gpfs/commons/projects/knowles_singlecell_splicing/TabulaSenis/data/AWS/processed_for_scanpy/tabulamurissenisfacsofficialrawobj.h5ad"
adata = sc.read_h5ad(exp_file)
# Reset the index of adata.obs to integers and drop the old index
adata.obs.reset_index(drop=True, inplace=True)
adata.var["mouse_gene_name"] = adata.var.index

In [11]:
# if age column is 3m, adjust cell_clean to replace the first "." with "_" and the second "." also with "_" 
adata.obs['cell_clean'] = adata.obs['cell']  # Start by copying the 'cell' column to 'cell_clean'
adata.obs.loc[adata.obs['age'] == '3m', 'cell_clean'] = adata.obs.loc[adata.obs['age'] == '3m', 'cell'].str.replace('.', '_', 2)
adata.obs["cell_clean"] = adata.obs["cell_clean"].str.split('_').str[:2].str.join('_')

In [12]:
# Make cell IDs comperable 
splice_adata = splice_adata[splice_adata.obs["cell_clean"].isin(adata.obs["cell_clean"])]
adata = adata[adata.obs["cell_clean"].isin(splice_adata.obs["cell_clean"])]
adata.obs.reset_index(drop=True, inplace=True)

# Rename the existing 'cell_id_index' to 'old_cell_id_index'
splice_adata.obs.rename(columns={'cell_id_index': 'old_cell_id_index'}, inplace=True)
splice_adata.obs.reset_index(inplace=True, drop=True)

# Create a new 'cell_id_index' based on the current index
splice_adata.obs['cell_id_index'] = splice_adata.obs.index
splice_adata.obs_names = splice_adata.obs["cell_clean"]
adata.obs_names = adata.obs["cell_clean"]

/scratch/ipykernel_1420882/2861903675.py:11: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  splice_adata.obs['cell_id_index'] = splice_adata.obs.index
/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [13]:
print(adata.shape)
print(splice_adata.shape)

(106199, 22966)
(106199, 71500)


#### Clean up tissue and cell_type grouping names

In [14]:
# Dictionary to consolidate duplicates
subtissue_corrections = {
    'T cells': 'T-cells',
    'ENDOMUCIN': 'Endomucin',
    'forelimb and hindlimb': 'ForelimbandHindlimb',
    'Liver non-hepato/SCs_st': 'Liver non-hepato/SCs',
    'Skin Anagen': 'Anagen'
}

cell_type_groupings = {
    
    # Basal cells
    'basal cell of epidermis': 'BASAL CELL',
    'basal cell': 'BASAL CELL',

    # Endothelial cells
    'endothelial cell': 'ENDOTHELIAL CELL',
    'endothelial cell of coronary artery': 'ENDOTHELIAL CELL',
    'endothelial cell of hepatic sinusoid': 'ENDOTHELIAL CELL',
    'aortic endothelial cell': 'ENDOTHELIAL CELL',
    'vein endothelial cell': 'ENDOTHELIAL CELL',
    'endothelial cell of lymphatic vessel': 'ENDOTHELIAL CELL',

    # T cells
    'T cell': 'T CELL',
    'CD4-positive, alpha-beta T cell': 'T CELL',
    'CD8-positive, alpha-beta T cell': 'T CELL',
    'regulatory T cell': 'T CELL',
    'mature NK T cell': 'T CELL',
    'mature alpha-beta T cell': 'T CELL',
    
    # B cells
    'B cell': 'B CELL',
    'immature B cell': 'B CELL',
    'naive B cell': 'B CELL',
    'precursor B cell': 'B CELL',
    'early pro-B cell': 'B CELL',
    'late pro-B cell': 'B CELL',
    'plasma cell': 'B CELL',

    # Fibroblasts
    'fibroblast': 'FIBROBLAST',
    'fibroblast of cardiac tissue': 'FIBROBLAST',
    'fibroblast of lung': 'FIBROBLAST',
    'pulmonary interstitial fibroblast': 'FIBROBLAST',
    'kidney interstitial fibroblast': 'FIBROBLAST',
    'fibrocyte': 'FIBROBLAST',

    # Macrophages 
    'macrophage': 'MACROPHAGE',
    'Kupffer cell': 'MACROPHAGE', # macrophages in the liver
    'lung macrophage': 'MACROPHAGE',

    # Monocytes
    'monocyte': 'MONOCYTE',
    'classical monocyte': 'MONOCYTE',
    'non-classical monocyte': 'MONOCYTE',
    'intermediate monocyte': 'MONOCYTE',

    # General Immune Cells
    'granulocyte': 'GRANULOCYTE',
    'basophil': 'GRANULOCYTE', 
    'granulocyte monocyte progenitor cell': 'GRANULOCYTE',

    'leukocyte': 'GENERAL IMMUNE CELL',
    'professional antigen presenting cell': 'ANTIGEN PRESENTING CELL',

    'lymphocyte': 'LYMPHOID IMMUNE CELL',
    'NK cell': 'LYMPHOID IMMUNE CELL',

    'myeloid cell': 'MYELOID IMMUNE CELL',
    'myeloid leukocyte': 'MYELOID IMMUNE CELL',
    'granulocytopoietic cell': 'MYELOID IMMUNE CELL',
    'promonocyte': 'MYELOID IMMUNE CELL',

    'thymocyte': 'THYMOCYTE',
    'DN4 thymocyte': 'THYMOCYTE',

    # Neutrophils
    'neutrophil': 'NEUTROPHIL',

    # Dendritic Cells
    'dendritic cell': 'DENDRITIC CELL',
    'plasmacytoid dendritic cell': 'DENDRITIC CELL',
    'myeloid dendritic cell': 'DENDRITIC CELL',

    # Microglia (Brain Immune Cells)
    'microglial cell': 'MICROGLIA',
    
    # Pancreatic cells
    'pancreatic A cell': 'PANCREATIC CELL',
    'pancreatic B cell': 'PANCREATIC CELL',
    'pancreatic D cell': 'PANCREATIC CELL',
    'pancreatic acinar cell': 'PANCREATIC CELL',
    'pancreatic PP cell': 'PANCREATIC CELL',
    'pancreatic ductal cell': 'PANCREATIC CELL',
    'pancreatic stellate cell': 'PANCREATIC CELL',
    
    # Smooth muscle cells
    'smooth muscle cell': 'SMOOTH MUSCLE CELL',
    'bronchial smooth muscle cell': 'SMOOTH MUSCLE CELL',
    'smooth muscle cell of the pulmonary artery': 'SMOOTH MUSCLE CELL',
    'smooth muscle cell of trachea': 'SMOOTH MUSCLE CELL',
    
    # Epithelial cells (includes luminal epithelial cell of mammary gland)
    'epithelial cell': 'EPITHELIAL CELL',
    'epidermal cell': 'EPITHELIAL CELL',
    'epithelial cell of large intestine': 'EPITHELIAL CELL',
    'enterocyte of epithelium of large intestine': 'EPITHELIAL CELL',
    'epithelial cell of proximal tubule': 'EPITHELIAL CELL',
    'epithelial cell of thymus': 'EPITHELIAL CELL',
    'bladder urothelial cell': 'EPITHELIAL CELL',
    'basal epithelial cell of tracheobronchial tree': 'EPITHELIAL CELL',
    'luminal epithelial cell of mammary gland': 'EPITHELIAL CELL',

    # Neurons
    'neuron': 'NEURON',
    'medium spiny neuron': 'NEURON',
    'interneuron': 'NEURON',
    'neuronal stem cell': 'NEURON',

    # Glial Cells (excluding microglia)
    'oligodendrocyte': 'GLIAL CELL',
    'oligodendrocyte precursor cell': 'GLIAL CELL',
    'astrocyte': 'GLIAL CELL',
    'Bergmann glial cell': 'GLIAL CELL',
    'ependymal cell': 'GLIAL CELL',

    # Stem cells
    'mesenchymal stem cell': 'STEM CELL',
    'mesenchymal stem cell of adipose': 'STEM CELL',
    'hematopoietic stem cell': 'STEM CELL',
    'neuronal stem cell': 'STEM CELL',
    'intestinal crypt stem cell': 'STEM CELL',
    'keratinocyte stem cell': 'STEM CELL',
    'lymphoid progenitor cell': 'STEM CELL',
    'proerythroblast': 'STEM CELL',
    'megakaryocyte-erythroid progenitor cell': 'STEM CELL',

    # Other specialized cells
    'ventricular myocyte': 'CARDIAC MUSCLE CELL',
    'atrial myocyte': 'CARDIAC MUSCLE CELL',
    'skeletal muscle satellite cell': 'SKELETAL MUSCLE CELL',
    'kidney collecting duct principal cell': 'KIDNEY CELL',
    'kidney collecting duct epithelial cell': 'KIDNEY CELL',
    'kidney interstitial fibroblast': 'FIBROBLAST',
    'mesangial cell': 'KIDNEY CELL',
    'type I pneumocyte': 'LUNG CELL',
    'type II pneumocyte': 'LUNG CELL',
    'club cell of bronchiole': 'LUNG CELL',
    'lung neuroendocrine cell': 'LUNG CELL',
    'ciliated columnar cell of tracheobronchial tree': 'LUNG CELL',
    'respiratory basal cell': 'LUNG CELL',
    'Brush cell of epithelium proper of large intestine': 'INTESTINAL CELL',
    'large intestine goblet cell': 'INTESTINAL CELL',
    'enteroendocrine cell': 'INTESTINAL CELL',
    'stromal cell': 'STROMAL CELL',
    'pericyte cell': 'PERICYTE',
    'brain pericyte': 'PERICYTE',
    'adventitial cell': 'STROMAL CELL',
    'keratinocyte': 'KERATINOCYTE',
    'bulge keratinocyte': 'KERATINOCYTE',
    'hepatocyte': 'HEPATOCYTE',
    'bladder cell': 'BLADDER CELL',
    'secretory cell': 'SECRETORY CELL',
    'endocardial cell': 'ENDOCARDIAL CELL',
    'valve cell': 'VALVE CELL',
    'chondrocyte': 'STROMAL CELL',
    'fenestrated cell': 'FENESTRATED CELL',
    'neuroepithelial cell': 'NEUROEPITHELIAL CELL',
    'kidney loop of Henle ascending limb epithelial cell': 'KIDNEY CELL',
    'mucus secreting cell': 'SECRETORY CELL'
}

In [15]:
# Remove any leading/trailing whitespace from subtissue values
splice_adata.obs['subtissue'] = splice_adata.obs['subtissue'].str.strip()
splice_adata.obs['subtissue_clean'] = splice_adata.obs['subtissue'].replace(subtissue_corrections)

# Drop the old subtissue 
splice_adata.obs.drop(columns=['subtissue'], inplace=True)

# Add new cell type groupings 
splice_adata.obs['cell_ontology_class'] = splice_adata.obs['cell_ontology_class'].astype(str)
splice_adata.obs['cell_type_grouped'] = splice_adata.obs['cell_ontology_class'].replace(cell_type_groupings)
splice_adata.obs['cell_type_grouped'] = splice_adata.obs['cell_type_grouped'].fillna(adata.obs['cell_ontology_class'])

#### Subset to just cell types with more than 50 cells in them

In [16]:
# Step 1: Count the number of cells per cell type in 'cell_ontology_class'
cell_type_counts = splice_adata.obs['cell_type_grouped'].value_counts()

# Step 2: Filter for cell types with more than 50 cells
cell_types_to_keep = cell_type_counts[cell_type_counts > 50].index

# Step 3: Subset the AnnData object to only include these cell types
splice_adata = splice_adata[splice_adata.obs['cell_type_grouped'].isin(cell_types_to_keep)]

# Print the subsetted cell types and their counts
print(splice_adata.obs['cell_type_grouped'].value_counts())

cell_type_grouped
MICROGLIA                  12796
STEM CELL                  10946
B CELL                     10221
ENDOTHELIAL CELL            8697
EPITHELIAL CELL             7715
FIBROBLAST                  5715
BASAL CELL                  5334
MYELOID IMMUNE CELL         4106
T CELL                      4063
KERATINOCYTE                3718
THYMOCYTE                   3327
GRANULOCYTE                 3222
PANCREATIC CELL             3004
GLIAL CELL                  2937
SKELETAL MUSCLE CELL        2682
MACROPHAGE                  2678
SMOOTH MUSCLE CELL          2274
INTESTINAL CELL             1886
MONOCYTE                    1602
HEPATOCYTE                  1152
BLADDER CELL                 939
LYMPHOID IMMUNE CELL         855
STROMAL CELL                 823
KIDNEY CELL                  805
NEURON                       794
SECRETORY CELL               589
CARDIAC MUSCLE CELL          542
PERICYTE                     519
DENDRITIC CELL               441
ENDOCARDIAL CELL         

In [17]:
# Filter the gene expression object to just these cells and add the useful columns 
adata_reordered = adata[splice_adata.obs_names]
splice_adata.obs.rename_axis('cell_id_for_index', inplace=True)
adata_reordered.obs.rename_axis('cell_id_for_index', inplace=True)

original_order = adata_reordered.obs.index
merged_obs = adata_reordered.obs.merge(splice_adata.obs, on=['cell_clean', 'age', 'method', 'mouse.id', 'cell_ontology_class', 'tissue', 'sex'])
merged_obs.set_index(original_order, inplace=True)
adata_reordered.obs = merged_obs

In [18]:
(adata_reordered.obs_names == splice_adata.obs_names).all()

True

### Processing gene expression data --> raw counts --> normalize

In [19]:
# Save raw counts as adata_reordered layer 
adata_reordered.layers["raw_counts"] = adata_reordered.X.copy()
adata_reordered.layers["raw_counts"].data

array([ 32., 108.,   2., ...,  24.,  39., 339.], dtype=float32)

In [20]:
# Step 1: Normalize the data and log transform it
sc.pp.normalize_total(adata_reordered, target_sum=1e4)
sc.pp.log1p(adata_reordered)

# Step 2: Identify highly variable genes
sc.pp.highly_variable_genes(adata_reordered, n_top_genes=2000, subset=False)  
print(f"Number of highly variable genes: {adata_reordered.shape[1]}")

# Step 3: Perform PCA
sc.pp.pca(adata_reordered, n_comps=50)  # Default is 50 principal components
print("PCA completed. Shape of the PCA matrix:", adata_reordered.obsm['X_pca'].shape)

# Step 4: Compute the neighborhood graph (needed for UMAP and clustering)
sc.pp.neighbors(adata_reordered, n_neighbors=10, n_pcs=40)  # Use 40 PCs

# Step 5: Perform UMAP
sc.tl.umap(adata_reordered)
print("UMAP completed. Shape of the UMAP embedding:", adata_reordered.obsm['X_umap'].shape)

# Step 6: Perform clustering (Leiden algorithm by default)
sc.tl.leiden(adata_reordered, resolution=1.0)  # Adjust the resolution to control cluster granularity
print("Leiden clustering completed. Number of clusters:", adata_reordered.obs['leiden'].nunique())

Number of highly variable genes: 22966
PCA completed. Shape of the PCA matrix: (106199, 50)
UMAP completed. Shape of the UMAP embedding: (106199, 2)


/scratch/ipykernel_1420882/1868906508.py:21: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata_reordered, resolution=1.0)  # Adjust the resolution to control cluster granularity


Leiden clustering completed. Number of clusters: 56


### Merge atse gene information if it's there!

In [21]:
if gtf_annot:
    splice_adata.var = splice_adata.var.merge(atses[["gene_id", "gene_name", "junction_id"]], on=["gene_id", "junction_id"])
else:
    splice_adata.var = splice_adata.var.merge(atses[["junction_id"]], on=["junction_id"])    

#### Obtain sparse junction usage ratios!

In [22]:
# Extract junction counts and cluster counts as sparse matrices
junction_counts = splice_adata.layers["cell_by_junction_matrix"] 
junction_counts = junction_counts.tocoo()

cluster_counts = splice_adata.layers["cell_by_cluster_matrix"]    
cluster_counts = cluster_counts.tocoo()

KeyboardInterrupt: 

In [ ]:
junc_norm_sum = wayp.sparse_sum(junction_counts, 0) / junction_counts.shape[0]

plt.hist(np.log1p(junc_norm_sum), 100)
plt.xlabel("log1p(mean junction count) across all cells for each junction")

min_junc_mean = 0.005 # default 
to_keep = junc_norm_sum > min_junc_mean

print(f"Number of junctions in matrix before filtering: {len(junc_norm_sum)}")
print(f"Number of junctions after filtering {len(junc_norm_sum[to_keep])}")

In [ ]:
juncs_filter = splice_adata.var[to_keep]
# Ensure we are only keeping ATSEs for downstream analysis with at least two junctions post filtering 
cluster_junction_counts = juncs_filter.groupby("Cluster")["junction_id"].count()
clusters_to_keep = cluster_junction_counts[cluster_junction_counts >= 2].index
juncs_filter_final = juncs_filter[juncs_filter["Cluster"].isin(clusters_to_keep)]
juncs_filter_final

In [ ]:
splice_adata.var['old_index'] = splice_adata.var.index
splice_adata = splice_adata[:, juncs_filter_final.index]
splice_adata.var = splice_adata.var.reset_index(drop=True)
splice_adata.var["junction_id_index"] = splice_adata.var.index

In [ ]:
# ---
# now make sure there are no cells wtih zero counts across the board!
# ---

# Access the sparse matrix with junction counts (assumed to be in layers['cell_by_junction_matrix'])
junction_matrix = splice_adata.layers['cell_by_junction_matrix']

# Check if the matrix is sparse (it should be sparse to avoid conversion to dense)
if isinstance(junction_matrix, csr_matrix):
    # Sum across each row (axis=1) and check if the sum is zero (meaning the row is all zeros)
    non_zero_cells = junction_matrix.getnnz(axis=1) > 0
else:
    raise ValueError("The matrix is not sparse!")

print(len(non_zero_cells))

# Filter the AnnData object to remove cells with all zero counts
splice_adata = splice_adata[non_zero_cells, :]

In [ ]:
splice_adata.var["Cluster"].value_counts().min(), splice_adata.var["Cluster"].value_counts().max()

In [ ]:
# Extract junction counts and cluster counts as sparse matrices (is this necessary???!)
junction_counts = splice_adata.layers["cell_by_junction_matrix"] 
cluster_counts = splice_adata.layers["cell_by_cluster_matrix"]    

# Ensure the matrices are in COO format for easier element-wise operations
junction_counts = junction_counts.tocoo()
cluster_counts = cluster_counts.tocoo()

In [ ]:
# juncs_clusts = splice_adata.var[["junction_id", "Cluster"]]
## Apply the transformation and use .loc to set the new column to avoid the warning
# juncs_clusts.loc[:, 'formatted'] = juncs_clusts.apply(wayp.transform_row, axis=1)
## Print the resulting formatted values
# juncs_clusts[['formatted']].to_csv("formatted_junctions_splice_girls_input.tsv", sep="\t", index=False, header=False)

#### Get sparse PCA, UMAP and Diffusion Map!

In [ ]:
# Get sparse centered PSI values 
splice_adata.layers["junc_ratio"] = wayp.calculate_centered_psi(junction_counts, cluster_counts)

# Step 1: Perform PCA using sparse data
n_components = 40  # Number of components to keep

n_iter = 5  # Increasing the number of iterations for better convergence
svd = TruncatedSVD(n_components=n_components, n_iter=n_iter, random_state=42)

# Fit and transform the junction ratio data (this gives U)
U = svd.fit_transform(splice_adata.layers["junc_ratio"])

# Get the singular values (S)
S = svd.singular_values_

# Multiply U by S to get U * S
# Need to scale U by the singular values S (broadcasted across columns)
U_by_S = U * S  # This scales each component in U by the corresponding singular value in S

# Store the PCA results (U * S) in the 'X_pca' field of .obsm (multi-dimensional)
splice_adata.obsm['X_pca'] = U_by_S

# Optionally, store explained variance ratio for future reference
splice_adata.uns['pca_explained_variance_ratio'] = svd.explained_variance_ratio_

In [ ]:
pca_result = U_by_S

# Step 2: Compute UMAP on the PCA-reduced data
sc.pp.neighbors(splice_adata, use_rep='X_pca')

# Step 3. Calculate UMAP 
sc.tl.umap(splice_adata)

# Step 4. Calculate the diffusion map
sc.tl.diffmap(splice_adata)

# The diffusion components are now stored in adata.obsm['X_diffmap']
diffmap_components = splice_adata.obsm['X_diffmap']
pca_components = splice_adata.obsm["X_pca"]

# Step 5. Run tSNE 
sc.tl.tsne(splice_adata)
tsne_components = splice_adata.obsm["X_tsne"]

In [ ]:
# Define possible number of waypoints to learn
n_waypoints_learn = [30, 50, 100]

# Placeholder to store waypoints and metacell dictionaries for each n_waypoints
waypoints_dict = {}
metacell_dicts = {}

# Parameters
num_components = 10  # Number of diffusion map components to consider
metacell_size = 50    # Number of nearest cells to assign to each waypoint

# Loop over different n_waypoints to generate waypoints and metacell assignments
for n_waypoints in n_waypoints_learn:

    print(f"Finding {n_waypoints} waypoints from the diffusion components!")
    random_seed = np.random.randint(0, 10000 + 1)  # Generate random seed
    
    # Max-min sampling to identify waypoints
    waypoints = wayp.max_min_sampling(pca_components, n_waypoints, num_components=num_components, seed=random_seed)
    
    # Store waypoints for this particular number of waypoints
    waypoints_dict[n_waypoints] = waypoints

    # Assign nearest cells to each waypoint (metacells)
    metacell_dict = wayp.assign_nearest_cells(waypoints, pca_components, num_nearest=metacell_size)
    
    # Store the metacell dictionary for this number of waypoints
    metacell_dicts[n_waypoints] = metacell_dict

#### Try looking at waypoints just using PC space...

In [ ]:
wayp.plot_PCA_with_waypoints(splice_adata, waypoints_dict, n_waypoints=50, waypoint_color='red', first_waypoint_color='blue', size=10)

In [ ]:
wayp.plot_tSNE_with_waypoints(splice_adata, waypoints_dict, n_waypoints=50, waypoint_color='red', first_waypoint_color='blue', size=10)

In [ ]:
wayp.plot_UMAP_with_waypoints(splice_adata, waypoints_dict, n_waypoints=50, waypoint_color='red', first_waypoint_color='blue', size=10)

In [ ]:
splice_adata.obs.tissue.value_counts()

In [ ]:
# Get the unique tissues in adata.obs['cell_ontology_class']
tissues = splice_adata.obs['cell_type_grouped'].unique()

# Set up grid dimensions: 6 rows, adjust columns to fit the number of tissues
n_tissues = len(tissues)
n_rows = 4  # Set to 6 rows
n_cols = int(np.ceil(n_tissues / n_rows))  # Calculate the number of columns needed

# Create a grid of subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 5))  # Adjust figure size as needed
axes = axes.flatten()  # Flatten axes to easily loop through them

# Plot each tissue's UMAP in its subplot without legend
for i, tissue in enumerate(tissues):
    ax = axes[i]
    
    # Plot UMAP for the specific tissue on the corresponding subplot without the legend
    sc.pl.umap(splice_adata, color="cell_type_grouped", groups=[tissue], show=False, size=10, alpha=0.7, ax=ax, legend_loc="none")
    
    # Set title for each subplot with the tissue name
    ax.set_title(tissue)

# Turn off unused axes if there are any
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Adjust layout for better spacing
plt.tight_layout()

# Show the combined grid of plots
plt.show()

In [ ]:
X = splice_adata.obsm['X_diffmap']  # Design matrix based on diffusion components
rho_hat = splice_adata.layers["junc_ratio"]

# Generate multiple initializations
psi_initializations, phi_initializations = wayp.generate_initializations(rho_hat, waypoints_dict, metacell_dicts, epsilon=0.001)

In [ ]:
# Loop through the waypoints_dict and corresponding initializations
for i, n_waypoints in enumerate(waypoints_dict.keys()):

    print(f"Adding waypoint based initializations to anndata for {n_waypoints} waypoints!")

    # Extract the corresponding psi and phi initializations
    psi = psi_initializations[i]
    phi = phi_initializations[i]

    # Convert psi and phi to torch tensors if needed
    psi = torch.tensor(psi)
    phi = torch.tensor(phi)

    # Convert to NumPy arrays if psi and phi are torch tensors (just in case)
    if isinstance(psi, torch.Tensor):
        psi = psi.cpu().numpy()  # Convert to NumPy array
    if isinstance(phi, torch.Tensor):
        phi = phi.cpu().numpy()  # Convert to NumPy array

    # Store psi in `adata.varm` and phi in `adata.obsm` with keys based on the number of waypoints
    splice_adata.varm[f'psi_init_{n_waypoints}_waypoints'] = psi  # Store psi with name 'psi_init_{n_waypoints}_waypoints'
    splice_adata.obsm[f'phi_init_{n_waypoints}_waypoints'] = phi  # Store phi with name 'phi_init_{n_waypoints}_waypoints'


In [ ]:
# Convert the 'junc_ratio' layer to a CSR matrix before saving
if isinstance(splice_adata.layers['junc_ratio'], coo_matrix):
    splice_adata.layers['junc_ratio'] = splice_adata.layers['junc_ratio'].tocsr()

In [ ]:
# Original file path
original_path = input_file  # Input your original file path

# Extract the directory and filename
base_directory = os.path.dirname(original_path)
original_filename = os.path.basename(original_path)

# Get the current date and time as a string (format: YYYYMMDD_HHMMSS)
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Replace the date and timestamp part with "with_initializations" and append the new timestamp
filename_parts = original_filename.split('_')

# Conditionally add "brain_only" to the file name if brain_only is True
if brain_only:
    new_filename = f"{'_'.join(filename_parts[:-2])}_with_initializations_brain_only_{timestamp}.h5ad"
else:
    new_filename = f"{'_'.join(filename_parts[:-2])}_NO_GTF_with_initializations_{timestamp}.h5ad"

# Create the new full file path
new_file_path = os.path.join(base_directory, new_filename)

In [ ]:
# Save the AnnData object with the new filename
splice_adata.write_h5ad(new_file_path, compression='gzip')

print(f"AnnData saved as {new_file_path}")

#### Prep data input for model training!

In [ ]:
splice_adata.layers["Cluster_Counts"] = coo_matrix(splice_adata.layers["cell_by_cluster_matrix"])
splice_adata.layers["Junction_Counts"]  = coo_matrix(splice_adata.layers["cell_by_junction_matrix"])

junction_layer="Junction_Counts"
cluster_layer="Cluster_Counts"

In [ ]:
float_type = {"device": device, "dtype": torch.float16}

device = float_type["device"]

# Extract cell indices and junction indices from the sparse junction layer
cell_index_array = np.array(splice_adata.layers[junction_layer].row)
junc_index_array = np.array(splice_adata.layers[junction_layer].col)

# Convert the row and col of the sparse matrix to torch tensors
cell_index_tensor = torch.tensor(cell_index_array, dtype=torch.int32, device=device)
junc_index_tensor = torch.tensor(junc_index_array, dtype=torch.int32, device=device)

# Convert the data of the sparse matrix to torch tensor
ycount_array = np.array(splice_adata.layers[junction_layer].data)
ycount = torch.tensor(ycount_array, **float_type)

# Create the ycount sparse matrix (Junction counts)
ycount_lookup = torch.sparse_coo_tensor(
        indices=torch.stack([cell_index_tensor, junc_index_tensor]), 
        values=ycount,
        size=(len(splice_adata.obs), len(splice_adata.var))
        ).to_sparse_csr()

# Extract the cluster layer (total counts)
coo2 = splice_adata.layers[cluster_layer]
total_counts_tensor = torch.tensor(coo2.data, **float_type)
    
# Create the tcount sparse matrix (Total counts)
tcount_lookup = torch.sparse_coo_tensor(
        indices=torch.stack([cell_index_tensor, junc_index_tensor]), 
        values=total_counts_tensor,
        size=(len(splice_adata.obs), len(splice_adata.var))
    ).to_sparse_csr()

# Convert sparse matrices to COO format, ensure they are on the GPU, and follow the specified float type
full_y_tensor = ycount_lookup.to_sparse_coo()
full_total_counts_tensor = tcount_lookup.to_sparse_coo()

# Define the path to save to
path_tosaveto = '/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/'

# Check if brain_only is True or False
if brain_only:
    # File names for brain-only tensors
    full_y_tensor_filename = 'full_y_tensor_brain_only.pt'
    full_total_counts_tensor_filename = 'full_total_counts_tensor_brain_only.pt'
else:
    # File names for the full tensors
    full_y_tensor_filename = 'full_y_tensor.pt'
    full_total_counts_tensor_filename = 'full_total_counts_tensor.pt'

# Create full paths by joining the directory with the filenames
full_y_tensor_path = os.path.join(path_tosaveto, full_y_tensor_filename)
full_total_counts_tensor_path = os.path.join(path_tosaveto, full_total_counts_tensor_filename)

# Save the sparse tensors to the specified files
torch.save(full_y_tensor, full_y_tensor_path)
torch.save(full_total_counts_tensor, full_total_counts_tensor_path)

# Print confirmation
print(f"full_y_tensor saved to {full_y_tensor_path}")
print(f"full_total_counts_tensor saved to {full_total_counts_tensor_path}")

In [ ]:
from datetime import datetime

In [ ]:
# Step 1: Get the current date in the desired format
current_date = datetime.now().strftime('%Y-%m-%d')

# Step 2: Format the file path with the current date
new_file_path = f"/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/TMS_Anndata_GeneExpression_{current_date}.h5ad"

# Step 3: Save the anndata object with gzip compression
adata_reordered.write_h5ad(new_file_path, compression='gzip')

print(f"Gene expression Anndata object successfully saved to {new_file_path}")